# Training entry point

The authoritative training logic lives in `src.train_price_model`; this notebook is only a manual runner.

In [11]:
import os
import sys

project_root = os.path.abspath("..")
src_dir = os.path.join(project_root, "src")
util_dir = os.path.join(project_root, "util")
for path in (project_root, src_dir, util_dir):
    if path not in sys.path:
        sys.path.insert(0, path)

%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from src.config import DATABASE_PATH, DEMAND_UPSTREAM_MODEL_PATH
from src.etl_price import create_price_tables
from src.train_predict_model import load_model_from_pickle
from src.train_price_model import evaluate_price_model_walk_forward, train_price_model

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import pandas as pd
from src.config import DATABASE_PATH, DEMAND_UPSTREAM_MODEL_PATH
from src.etl_price import create_price_tables
from src.train_predict_model import load_model_from_pickle
from src.train_price_model import evaluate_price_model_walk_forward, train_price_model

model, params, path = train_price_model(
    model_family="lightgbm",
    train_end="2026-03-31",
    validation_end="2026-04-30",
)
print("Best parameters:", params)
print("Production model:", path)

connection = create_price_tables(DATABASE_PATH)
demand_model = load_model_from_pickle(DEMAND_UPSTREAM_MODEL_PATH)

# evaluate the model using walk-forward validation
scores = evaluate_price_model_walk_forward(
    model=model,
    connection=connection,
    demand_model=demand_model,
    start="2026-05-01",
    end="2026-08-01",
    model_family="lightgbm",
)
print("Walk-forward scores:", scores)



---------- Updating price database ----------

Done with fetching and storing SMARD price/generation series: 2026-08-20 -> 2026-08-21

Open-Meteo weather is up to date (target: yesterday).

---------- Updating price database done ----------
---------- Updating demand database ----------
Database ready: d:\Projects\DataScience\Portfolio\electricity_price_forecast\db\energy_demand.db

[Energy] Up to date — nothing to do.

[Weather] Up to date — nothing to do.

---------- Updating demand database done ----------
[LightGBM] [Info] Total Bins 8208
[LightGBM] [Info] Number of data points in the train set: 63138, number of used features: 37
[LightGBM] [Info] Start training from score 95.299848
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Li

In [13]:
# train and evaluate XGBoost model
model, params, path = train_price_model(
    model_family="xgboost",
    train_end="2026-03-31",
    validation_end="2026-04-30",
)
print("Best parameters:", params)
print("Production model:", path)

connection = create_price_tables(DATABASE_PATH)
demand_model = load_model_from_pickle(DEMAND_UPSTREAM_MODEL_PATH)

# evaluate the model using walk-forward validation
scores = evaluate_price_model_walk_forward(
    model=model,
    connection=connection,
    demand_model=demand_model,
    start="2026-05-01",
    end="2026-08-01",
    model_family="xgboost",
)
print("Walk-forward scores:", scores)



---------- Updating price database ----------

Done with fetching and storing SMARD price/generation series: 2026-08-20 -> 2026-08-21

Open-Meteo weather is up to date (target: yesterday).

---------- Updating price database done ----------
---------- Updating demand database ----------
Database ready: d:\Projects\DataScience\Portfolio\electricity_price_forecast\db\energy_demand.db

[Energy] Up to date — nothing to do.

[Weather] Up to date — nothing to do.

---------- Updating demand database done ----------
Best parameters: OrderedDict({'learning_rate': 0.085601396371112, 'max_depth': 3, 'n_estimators': 1000})
Production model: d:\Projects\DataScience\Portfolio\electricity_price_forecast\models\production\price_xgboost.pkl
Walk-forward scores: {'mae': 18.62197592302401, 'rmse': 35.04136718917843, 'r2': 0.7504640673093472, 'n_test': 2208.0, 'input_lineage_records': 552.0, 'weather_fallback_targets': 3.0, 'weather_max_fallback_hours': 12.0, 'input_unavailable_records': 0.0}
